In [1]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv('source/diabetes.csv')

In [3]:
data

,Number of times pregnant,Plasma glucose concentration,Diastolic blood pressure,Triceps skin fold thickness,2-Hour serum insulin,Body mass index,Age,Class
0,6,148,72,35,0,33.6,50,positive
1,1,85,66,29,0,26.6,31,negative
2,8,183,64,0,0,23.3,32,positive
3,1,89,66,23,94,28.1,21,negative
4,0,137,40,35,168,43.1,33,positive
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,63,negative
764,2,122,70,27,0,36.8,27,negative
765,5,121,72,23,112,26.2,30,negative
766,1,126,60,0,0,30.1,47,positive


In [4]:
x = data.drop('Class', axis=1).values
y_string = data['Class'].values

In [5]:
y = np.empty(len(y_string), dtype='float64')
for i in range(len(y_string)):
    if y_string[i] == 'positive':
        y[i] = 1
    else:
        y[i] = 0

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3)

In [7]:
scaler = StandardScaler()
scaler.fit(x_train)
x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)

In [8]:
x_train = torch.tensor(x_train)
y_train = torch.tensor(y_train).view(-1, 1)
x_test = torch.tensor(x_test)
y_test = torch.tensor(y_test).view(-1, 1)

In [9]:
class Dataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return len(self.x)

In [10]:
dataset = Dataset(x_train, y_train)

In [11]:
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [12]:
print("There is {} batches in the dataset".format(len(train_loader)))
for (x, y) in train_loader:
    print('For one iteration (batch), there is:')
    print('Data:       {}'.format(x.shape))
    print('Labels:     {}'.format(y.shape))
    break

There is 17 batches in the dataset
For one iteration (batch), there is:
Data:       torch.Size([32, 7])
Labels:     torch.Size([32, 1])


In [13]:
# будем делать модель, в которой 7 нейронов на входе
# потом 3 скрытых слоя из (5, 4, 3) нейронов
# на выходе остаётся 1 нейрон
# используем сигмоиду с трешхолдом в самом конце

In [14]:
class Model(nn.Module):
    def __init__(self, input_features, output_feautres):
        super().__init__()
        self.fc1 =  nn.Linear(input_features, 5)
        self.fc2 = nn.Linear(5, 4)
        self.fc3 = nn.Linear(4, 3)
        self.fc4 = nn.Linear(3, output_feautres)
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()
    def forward(self, x):
        out = self.fc1(x)
        out = self.tanh(out)
        out = self.fc2(out)
        out = self.tanh(out)
        out = self.fc3(out)
        out = self.tanh(out)
        out = self.fc4(out)
        out = self.sigmoid(out)
        return out

In [15]:
net = Model(7, 1)
criterion = nn.BCELoss(reduction='mean')
optimizer = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9)

In [16]:
epochs = 200
for epoch in range(epochs):
    for inputs, labels in train_loader:
        inputs = inputs.float()
        labels = labels.float()
        # Forward
        outputs = net.forward(inputs)
        # Loss
        loss = criterion(outputs, labels)
        # Clear Grad Buffer
        optimizer.zero_grad()
        # Backprop
        loss.backward()
        # Update weights (w = w - lr * grad)
        optimizer.step()
    # Accuracy Calculation
    output = (outputs > 0.3).float()
    accuracy = (output == labels).float().mean()
    print(f"Epoch {epoch + 1} / {epochs} ")
    print(f"Loss: {loss:.3f} ")
    print(f"Accuracy: {accuracy:.3f}\n")

Epoch 1 / 200 
Loss: 0.533 
Accuracy: 0.800

Epoch 2 / 200 
Loss: 0.509 
Accuracy: 0.720

Epoch 3 / 200 
Loss: 0.369 
Accuracy: 0.720

Epoch 4 / 200 
Loss: 0.506 
Accuracy: 0.560

Epoch 5 / 200 
Loss: 0.586 
Accuracy: 0.640

Epoch 6 / 200 
Loss: 0.533 
Accuracy: 0.800

Epoch 7 / 200 
Loss: 0.610 
Accuracy: 0.680

Epoch 8 / 200 
Loss: 0.386 
Accuracy: 0.600

Epoch 9 / 200 
Loss: 0.408 
Accuracy: 0.720

Epoch 10 / 200 
Loss: 0.444 
Accuracy: 0.720

Epoch 11 / 200 
Loss: 0.335 
Accuracy: 0.880

Epoch 12 / 200 
Loss: 0.592 
Accuracy: 0.720

Epoch 13 / 200 
Loss: 0.545 
Accuracy: 0.680

Epoch 14 / 200 
Loss: 0.603 
Accuracy: 0.760

Epoch 15 / 200 
Loss: 0.451 
Accuracy: 0.880

Epoch 16 / 200 
Loss: 0.691 
Accuracy: 0.600

Epoch 17 / 200 
Loss: 0.388 
Accuracy: 0.840

Epoch 18 / 200 
Loss: 0.388 
Accuracy: 0.800

Epoch 19 / 200 
Loss: 0.551 
Accuracy: 0.760

Epoch 20 / 200 
Loss: 0.462 
Accuracy: 0.640

Epoch 21 / 200 
Loss: 0.349 
Accuracy: 0.760

Epoch 22 / 200 
Loss: 0.616 
Accuracy: 0.76

In [17]:
x_test = x_test.float()
y_test = y_test.float()
outputs = net.forward(x_test)
output = (outputs > 0.3).float()
accuracy = (output == y_test).float().mean()
print(f"Accuracy: {accuracy:.3f}\n")

Accuracy: 0.745

